In [29]:
import pandas as pd
import numpy as np

file_path = 'multi_platform_social_sentiment_evolution.csv'

try:
    df = pd.read_csv(file_path)
    print("--- Đọc file thành công! ---")
except FileNotFoundError:
    print("Không tìm thấy file. Hãy kiểm tra lại đường dẫn.")

print("\nThông tin Dataset:")
print(df.info())


--- Đọc file thành công! ---

Thông tin Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 31 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   post_id                           150000 non-null  str    
 1   platform                          150000 non-null  str    
 2   timestamp                         150000 non-null  str    
 3   date                              150000 non-null  str    
 4   hour_of_day                       150000 non-null  int64  
 5   day_of_week                       150000 non-null  int64  
 6   is_weekend                        150000 non-null  int64  
 7   user_id                           150000 non-null  str    
 8   followers                         150000 non-null  int64  
 9   account_age_days                  150000 non-null  int64  
 10  verified                          150000 non-null  int64  
 11  topic         

In [30]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Extract additional time features (optional)
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['month'] = df['timestamp'].dt.month

In [31]:
df['log_followers'] = np.log1p(df['followers'])

In [32]:
# 6. DEFINE FEATURE GROUP
categorical_cols = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

numerical_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]


In [33]:
# Target chính
df['target_engagement'] = df['engagement_rate_per_1k_followers']

In [34]:
# dùng percentile để chia high / low engagement
threshold = df['target_engagement'].quantile(0.8)

df['label'] = (df['target_engagement'] > threshold).astype(int)

print("Threshold:", threshold)
print(df['label'].value_counts())

Threshold: 40.0
label
0    120012
1     29988
Name: count, dtype: int64


In [35]:
leakage_cols = [
    'likes',
    'shares',
    'comments',
    'total_engagement',
    'views',
    'engagement_rate_per_1k_followers',
    'viral_coefficient',
    'cross_platform_spread'
]

df = df.drop(columns=leakage_cols)

print("Remaining columns:", df.columns)

Remaining columns: Index(['post_id', 'platform', 'timestamp', 'date', 'hour_of_day',
       'day_of_week', 'is_weekend', 'user_id', 'followers', 'account_age_days',
       'verified', 'topic', 'language', 'content_length', 'media_type',
       'num_hashtags', 'sentiment_category', 'sentiment_positive',
       'sentiment_negative', 'sentiment_neutral', 'hours_since_post',
       'toxicity_score', 'location', 'hour', 'day', 'month', 'log_followers',
       'target_engagement', 'label'],
      dtype='str')


In [36]:
# Thêm 'cross_platform_spread' vào danh sách cần xóa
leakage_cols = ['label', 'target_engagement']

# Chạy lại bước chia X, y
X = df.drop(columns=leakage_cols, errors='ignore')
y = df['label']

# Sau đó chạy lại toàn bộ flow: Split -> Merge Gephi -> Preprocess -> Train

In [37]:
# 3. Chia tập Train/Test bằng GroupShuffleSplit theo user_id
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['user_id']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"\n--- Đã chia dữ liệu xong ---")
print(f"Tập Train: {len(df_train)} dòng")
print(f"Tập Test: {len(df_test)} dòng")


--- Đã chia dữ liệu xong ---
Tập Train: 120070 dòng
Tập Test: 29930 dòng


In [38]:

# -----------------------------------
# 1. COPY TRAIN DATA
# -----------------------------------
df_post = df_train.copy()

# post_id đã có sẵn
df_post["post_id"] = df_post["post_id"].astype(str)
# -----------------------------------
# 2. TIME COLUMN
# ưu tiên dùng timestamp
# -----------------------------------
df_post["time_dt"] = pd.to_datetime(df_post["timestamp"], errors="coerce")

# fallback nếu timestamp lỗi
mask = df_post["time_dt"].isna()

df_post.loc[mask, "time_dt"] = (
    pd.to_datetime(df_post.loc[mask, "date"], errors="coerce") +
    pd.to_timedelta(df_post.loc[mask, "hour_of_day"], unit="h")
)

# bỏ row lỗi thời gian
df_post = df_post.dropna(subset=["time_dt"])

# -----------------------------------
# 3. SORT
# -----------------------------------
df_post = df_post.sort_values("time_dt").reset_index(drop=True)

In [39]:
import networkx as nx
# 4. CREATE GRAPH
# -----------------------------------
G_post = nx.Graph()
# add nodes
G_post.add_nodes_from(df_post["post_id"])

# -----------------------------------
# PARAMETER
# -----------------------------------
TIME_WINDOW_HOURS = 3   # test 1 / 3 / 6

# -----------------------------------
# 5. BUILD EDGES
# same topic + same platform + near time
# -----------------------------------
groups = df_post.groupby(["topic", "platform"])

print("Building Post Temporal Graph...")

for (topic, platform), group in groups:

    group = group.sort_values("time_dt")

    post_ids = group["post_id"].values
    times = group["time_dt"].values

    n = len(group)

    for i in range(n):
        j = i + 1

        while j < n:

            diff_hours = (times[j] - times[i]) / pd.Timedelta(hours=1)

            if diff_hours <= TIME_WINDOW_HOURS:

                p1 = post_ids[i]
                p2 = post_ids[j]

                if G_post.has_edge(p1, p2):
                    G_post[p1][p2]["weight"] += 1
                else:
                    G_post.add_edge(p1, p2, weight=1)

                j += 1

            else:
                break

print("Done!")
print("Nodes:", G_post.number_of_nodes())
print("Edges:", G_post.number_of_edges())

Building Post Temporal Graph...
Done!
Nodes: 120070
Edges: 333056


In [40]:
# ==========================================
# NODE2VEC FOR POST GRAPH (G_post)
# Strongest quick test
# ==========================================

import pandas as pd
from node2vec import Node2Vec

# ----------------------------------
# 1. TRAIN NODE2VEC
# ----------------------------------
# Fast test config first
node2vec = Node2Vec(
    G_post,
    dimensions=16,      # thử 16 trước, ngon thì lên 32
    walk_length=10,
    num_walks=10,
    workers=4
)

model = node2vec.fit(
    window=5,
    min_count=1,
    batch_words=256
)

print("Node2Vec training done!")

# ----------------------------------
# 2. EXTRACT EMBEDDINGS
# ----------------------------------
embeddings = []

for node in G_post.nodes():
    vec = model.wv[str(node)]
    embeddings.append([node] + list(vec))

# ----------------------------------
# 3. TO DATAFRAME
# ----------------------------------
cols = ["post_id"] + [f"post_emb_{i}" for i in range(16)]

df_post_emb = pd.DataFrame(embeddings, columns=cols)

print(df_post_emb.shape)
print(df_post_emb.head())

# ----------------------------------
# 4. MERGE TO TRAIN / TEST
# ----------------------------------
df_train = df_train.merge(df_post_emb, on="post_id", how="left")
df_test  = df_test.merge(df_post_emb, on="post_id", how="left")

# fill missing
emb_cols = [f"post_emb_{i}" for i in range(16)]

df_train[emb_cols] = df_train[emb_cols].fillna(0)
df_test[emb_cols]  = df_test[emb_cols].fillna(0)

print("Merge complete!")

# ----------------------------------
# 5. ADD TO NUMERICAL COLS
# ----------------------------------
numerical_cols = numerical_cols + emb_cols

print("Ready for XGBoost!")

Computing transition probabilities:   0%|          | 0/120070 [00:00<?, ?it/s]

Node2Vec training done!
(120070, 17)
             post_id  post_emb_0  post_emb_1  post_emb_2  post_emb_3  \
0  TIK20250419000000    0.008208   -0.060086    0.043488    0.001125   
1  RED20250419091149    0.035330   -0.005842   -0.010829    0.013613   
2  INS20250419091150   -0.012358    0.003290   -0.054900    0.034150   
3  TWI20250419000001    0.099737   -0.782079    0.453125    1.033794   
4  INS20250419091151   -0.038620   -0.013476    0.039892    0.053037   

   post_emb_4  post_emb_5  post_emb_6  post_emb_7  post_emb_8  post_emb_9  \
0    0.024779    0.021276   -0.013936   -0.018442   -0.005646   -0.036587   
1   -0.036951    0.017967    0.018593   -0.045572   -0.053997   -0.049832   
2    0.006005   -0.003712    0.024517    0.049097   -0.041047   -0.045805   
3    0.841616    0.978681    0.367091   -0.285367   -1.854890   -1.015167   
4   -0.055037    0.058687   -0.007734    0.060825    0.020603   -0.002175   

   post_emb_10  post_emb_11  post_emb_12  post_emb_13  post_emb_14 

In [41]:
# ----------------------------------
# 1. FEATURE LIST
# ----------------------------------
post_emb_cols = [f"post_emb_{i}" for i in range(16)]

# đảm bảo không bị trùng cột
numerical_final = list(dict.fromkeys(numerical_cols + post_emb_cols))

categorical_final = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

In [44]:
# ----------------------------------
# 2. X / y
# ----------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
X_train = df_train[numerical_final + categorical_final]
X_test  = df_test[numerical_final + categorical_final]

y_train = df_train["label"]
y_test  = df_test["label"]

# ----------------------------------
# 3. PREPROCESSOR
# ----------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_final),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_final)
    ]
)

In [45]:
# ----------------------------------
# 4. CLASS IMBALANCE
# ----------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

In [47]:
# ----------------------------------
# 5. MODEL
# ----------------------------------
from xgboost import XGBClassifier
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

In [48]:
# ----------------------------------
# 6. PIPELINE
# ----------------------------------
from sklearn.pipeline import Pipeline
clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", model)
])

# ----------------------------------
# 7. TRAIN
# ----------------------------------
clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [50]:
# ----------------------------------
# 8. PREDICT
# ----------------------------------
from sklearn.metrics import classification_report, roc_auc_score
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# ----------------------------------
# 9. RESULT
# ----------------------------------
print("===== Classification Report =====")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc, 4))

===== Classification Report =====
              precision    recall  f1-score   support

           0       0.84      0.39      0.53     24005
           1       0.22      0.70      0.34      5925

    accuracy                           0.45     29930
   macro avg       0.53      0.55      0.43     29930
weighted avg       0.72      0.45      0.49     29930

ROC-AUC Score: 0.5663


In [51]:
# =====================================================
# BUILD TEMPORAL FEATURES FOR SOCIAL POPULARITY DATASET
# Copy-paste run directly
# =====================================================

import pandas as pd
import numpy as np

# ---------------------------------
# 1. Ensure timestamp datetime
# ---------------------------------
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# ---------------------------------
# 2. Basic time features
# ---------------------------------
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["weekday"] = df["timestamp"].dt.weekday

df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)

# ---------------------------------
# 3. Cyclical encoding (very useful)
# ---------------------------------
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# ---------------------------------
# 4. Recency features
# ---------------------------------
df["log_hours_since_post"] = np.log1p(df["hours_since_post"])

# ---------------------------------
# 5. Engagement velocity (POWERFUL)
# Only run if these columns exist
# ---------------------------------
if "likes" in df.columns:
    df["likes_per_hour"] = df["likes"] / (df["hours_since_post"] + 1)

if "views" in df.columns:
    df["views_per_hour"] = df["views"] / (df["hours_since_post"] + 1)

if "shares" in df.columns:
    df["shares_per_hour"] = df["shares"] / (df["hours_since_post"] + 1)

if "comments" in df.columns:
    df["comments_per_hour"] = df["comments"] / (df["hours_since_post"] + 1)

# ---------------------------------
# 6. Peak hour indicator
# Example: 18h -> 23h
# ---------------------------------
df["is_peak_hour"] = df["hour"].between(18, 23).astype(int)

# ---------------------------------
# 7. Work-hour indicator
# ---------------------------------
df["is_work_hour"] = df["hour"].between(9, 17).astype(int)

# ---------------------------------
# 8. Night posting
# ---------------------------------
df["is_night"] = ((df["hour"] >= 0) & (df["hour"] <= 5)).astype(int)

# ---------------------------------
# 9. Topic trend last day (simple proxy)
# ---------------------------------
# count posts same topic same date
df["topic_daily_count"] = df.groupby(["topic", "date"])["post_id"].transform("count")

# ---------------------------------
# 10. User activity proxy
# ---------------------------------
df["user_daily_posts"] = df.groupby(["user_id", "date"])["post_id"].transform("count")

print("Temporal features built successfully!")

# ---------------------------------
# 11. Suggested numerical cols to add
# ---------------------------------
temporal_cols = [
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
    "month_sin", "month_cos",
    "log_hours_since_post",
    "is_peak_hour",
    "is_work_hour",
    "is_night",
    "topic_daily_count",
    "user_daily_posts"
]

print("Use these cols:")
print(temporal_cols)

Temporal features built successfully!
Use these cols:
['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'log_hours_since_post', 'is_peak_hour', 'is_work_hour', 'is_night', 'topic_daily_count', 'user_daily_posts']


In [53]:
# ---------------------------------
# Temporal columns
# ---------------------------------
temporal_cols = [
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'month_sin', 'month_cos',
    'log_hours_since_post',
    'is_peak_hour',
    'is_work_hour',
    'is_night',
    'topic_daily_count',
    'user_daily_posts'
]

# ---------------------------------
# Numerical columns
# ---------------------------------
numerical_final = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
] + temporal_cols

# remove duplicate
numerical_final = list(dict.fromkeys(numerical_final))

# ---------------------------------
# Categorical columns
# ---------------------------------
categorical_final = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

In [56]:
temporal_cols = [
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'month_sin', 'month_cos',
    'log_hours_since_post',
    'is_peak_hour',
    'is_work_hour',
    'is_night',
    'topic_daily_count',
    'user_daily_posts'
]

merge_cols = ['post_id'] + temporal_cols

df_train = df_train.merge(df[merge_cols], on='post_id', how='left')
df_test = df_test.merge(df[merge_cols], on='post_id', how='left')

df_train[temporal_cols] = df_train[temporal_cols].fillna(0)
df_test[temporal_cols] = df_test[temporal_cols].fillna(0)

In [57]:
X_train = df_train[numerical_final + categorical_final]
X_test  = df_test[numerical_final + categorical_final]

y_train = df_train['label']
y_test  = df_test['label']


In [58]:
# ---------------------------------
# Preprocess
# ---------------------------------
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_final),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_final)
])

# ---------------------------------
# Imbalance
# ---------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()


In [59]:
# ---------------------------------
# Model
# ---------------------------------
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=neg/pos,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)


In [60]:
# ---------------------------------
# Pipeline
# ---------------------------------
clf = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

# ---------------------------------
# Train
# ---------------------------------
clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [61]:
# ---------------------------------
# Predict
# ---------------------------------
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

# ---------------------------------
# Evaluate
# ---------------------------------
print("===== Classification Report =====")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc,4))

===== Classification Report =====
              precision    recall  f1-score   support

           0       0.83      0.52      0.64     24005
           1       0.22      0.55      0.32      5925

    accuracy                           0.53     29930
   macro avg       0.52      0.54      0.48     29930
weighted avg       0.71      0.53      0.58     29930

ROC-AUC Score: 0.5616


In [62]:
# =====================================================
# FINAL HYBRID MODEL
# Metadata + Temporal + Post Node2Vec
# =====================================================

import numpy as np
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# -------------------------------------------------
# 1. POST EMBEDDING COLS
# -------------------------------------------------
post_emb_cols = [f'post_emb_{i}' for i in range(16)]

# -------------------------------------------------
# 2. TEMPORAL COLS
# -------------------------------------------------
temporal_cols = [
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'month_sin', 'month_cos',
    'log_hours_since_post',
    'is_peak_hour',
    'is_work_hour',
    'is_night',
    'topic_daily_count',
    'user_daily_posts'
]

# -------------------------------------------------
# 3. BASE NUMERICAL COLS
# -------------------------------------------------
base_num_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]

# -------------------------------------------------
# 4. COMBINE ALL NUMERICAL FEATURES
# -------------------------------------------------
numerical_final = (
    base_num_cols
    + temporal_cols
    + post_emb_cols
)

# remove duplicate safely
numerical_final = list(dict.fromkeys(numerical_final))

# -------------------------------------------------
# 5. CATEGORICAL COLS
# -------------------------------------------------
categorical_final = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

# -------------------------------------------------
# 6. Fill missing values
# -------------------------------------------------
for col in post_emb_cols + temporal_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(0)
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(0)

# -------------------------------------------------
# 7. X / y
# -------------------------------------------------
X_train = df_train[numerical_final + categorical_final]
X_test  = df_test[numerical_final + categorical_final]

y_train = df_train["label"]
y_test  = df_test["label"]

# -------------------------------------------------
# 8. Preprocess
# -------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_final),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_final)
    ]
)

# -------------------------------------------------
# 9. Class imbalance
# -------------------------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

# -------------------------------------------------
# 10. XGBoost
# -------------------------------------------------
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

# -------------------------------------------------
# 11. Pipeline
# -------------------------------------------------
clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", model)
])

# -------------------------------------------------
# 12. Train
# -------------------------------------------------
clf.fit(X_train, y_train)

# -------------------------------------------------
# 13. Predict
# -------------------------------------------------
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# -------------------------------------------------
# 14. Result
# -------------------------------------------------
print("===== FINAL HYBRID MODEL =====")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc, 4))

===== FINAL HYBRID MODEL =====
              precision    recall  f1-score   support

           0       0.85      0.34      0.48     24005
           1       0.22      0.75      0.34      5925

    accuracy                           0.42     29930
   macro avg       0.53      0.55      0.41     29930
weighted avg       0.72      0.42      0.46     29930

ROC-AUC Score: 0.5624


In [63]:
# ==========================================================
# DYNAMIC PAGERANK ON G_post + MERGE + XGBOOST
# Recency-weighted PageRank for Post Temporal Graph
# ==========================================================

import pandas as pd
import numpy as np
import networkx as nx

# ----------------------------------------------------------
# 1. COPY GRAPH
# ----------------------------------------------------------
G_dyn = G_post.copy()

# ----------------------------------------------------------
# 2. BUILD post_id -> timestamp MAP
# ----------------------------------------------------------
time_map = df_train.set_index("post_id")["timestamp"].to_dict()

# convert datetime safely
for k in time_map:
    time_map[k] = pd.to_datetime(time_map[k], errors="coerce")

# latest time in train
max_time = max([t for t in time_map.values() if pd.notna(t)])

# ----------------------------------------------------------
# 3. RECENCY WEIGHT ON EDGES
# newer posts => stronger influence
# ----------------------------------------------------------
LAMBDA = 0.05   # test 0.03 / 0.05 / 0.1

for u, v in G_dyn.edges():

    tu = time_map.get(u, max_time)
    tv = time_map.get(v, max_time)

    newest = max(tu, tv)

    delta_hours = (max_time - newest).total_seconds() / 3600

    decay = np.exp(-LAMBDA * delta_hours)

    base_w = G_dyn[u][v].get("weight", 1)

    G_dyn[u][v]["weight"] = base_w * decay

# ----------------------------------------------------------
# 4. COMPUTE PAGERANK
# ----------------------------------------------------------
print("Running Dynamic PageRank...")

pr = nx.pagerank(
    G_dyn,
    alpha=0.85,
    weight="weight",
    max_iter=100
)

print("Done!")

# ----------------------------------------------------------
# 5. TO DATAFRAME
# ----------------------------------------------------------
df_pr = pd.DataFrame({
    "post_id": list(pr.keys()),
    "dynamic_pagerank": list(pr.values())
})

# ----------------------------------------------------------
# 6. MERGE TO TRAIN / TEST
# ----------------------------------------------------------
df_train = df_train.merge(df_pr, on="post_id", how="left")
df_test  = df_test.merge(df_pr, on="post_id", how="left")

df_train["dynamic_pagerank"] = df_train["dynamic_pagerank"].fillna(0)
df_test["dynamic_pagerank"]  = df_test["dynamic_pagerank"].fillna(0)

# ----------------------------------------------------------
# 7. ADD FEATURE
# ----------------------------------------------------------
numerical_final = list(dict.fromkeys(numerical_final + ["dynamic_pagerank"]))

# ----------------------------------------------------------
# 8. TRAIN XGBOOST
# ----------------------------------------------------------
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

categorical_final = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

X_train = df_train[numerical_final + categorical_final]
X_test  = df_test[numerical_final + categorical_final]

y_train = df_train["label"]
y_test  = df_test["label"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_final),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_final)
])

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=neg/pos,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

clf = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

clf.fit(X_train, y_train)

# ----------------------------------------------------------
# 9. EVALUATE
# ----------------------------------------------------------
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

print("===== Dynamic PageRank Model =====")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc,4))

Running Dynamic PageRank...
Done!
===== Dynamic PageRank Model =====
              precision    recall  f1-score   support

           0       0.85      0.36      0.51     24005
           1       0.22      0.74      0.34      5925

    accuracy                           0.44     29930
   macro avg       0.53      0.55      0.42     29930
weighted avg       0.72      0.44      0.47     29930

ROC-AUC Score: 0.5656


In [64]:
# ==========================================================
# FINAL BEST COMBO MODEL
# Metadata + Post Node2Vec + Dynamic PageRank
# ==========================================================

import numpy as np
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# ----------------------------------------------------------
# 1. NODE2VEC EMBEDDING COLS
# ----------------------------------------------------------
post_emb_cols = [f'post_emb_{i}' for i in range(16)]

# ----------------------------------------------------------
# 2. BASE NUMERICAL FEATURES
# ----------------------------------------------------------
base_num_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]

# ----------------------------------------------------------
# 3. COMBINE FEATURES
# ----------------------------------------------------------
numerical_final = (
    base_num_cols
    + post_emb_cols
    + ['dynamic_pagerank']
)

# remove duplicates
numerical_final = list(dict.fromkeys(numerical_final))

# ----------------------------------------------------------
# 4. CATEGORICAL FEATURES
# ----------------------------------------------------------
categorical_final = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

# ----------------------------------------------------------
# 5. FILL MISSING VALUES
# ----------------------------------------------------------
for col in post_emb_cols + ['dynamic_pagerank']:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(0)
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(0)

# ----------------------------------------------------------
# 6. TRAIN / TEST
# ----------------------------------------------------------
X_train = df_train[numerical_final + categorical_final]
X_test  = df_test[numerical_final + categorical_final]

y_train = df_train["label"]
y_test  = df_test["label"]

# ----------------------------------------------------------
# 7. PREPROCESS
# ----------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_final),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_final)
    ]
)

# ----------------------------------------------------------
# 8. CLASS IMBALANCE
# ----------------------------------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

# ----------------------------------------------------------
# 9. MODEL
# ----------------------------------------------------------
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

# ----------------------------------------------------------
# 10. PIPELINE
# ----------------------------------------------------------
clf = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

# ----------------------------------------------------------
# 11. TRAIN
# ----------------------------------------------------------
clf.fit(X_train, y_train)

# ----------------------------------------------------------
# 12. PREDICT
# ----------------------------------------------------------
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

# ----------------------------------------------------------
# 13. RESULT
# ----------------------------------------------------------
print("===== FINAL BEST COMBO MODEL =====")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc,4))

===== FINAL BEST COMBO MODEL =====
              precision    recall  f1-score   support

           0       0.85      0.31      0.46     24005
           1       0.22      0.78      0.34      5925

    accuracy                           0.40     29930
   macro avg       0.54      0.55      0.40     29930
weighted avg       0.73      0.40      0.43     29930

ROC-AUC Score: 0.5655


In [ ]:
# ==========================================================
# FULL GCN FOR POST GRAPH (PyTorch Geometric)
# Predict label on post nodes
# ==========================================================

# pip install torch torchvision torchaudio
# pip install torch-geometric

import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv

# ==========================================================
# 1. USE TRAIN + TEST TOGETHER (nodes = all posts)
# labels only evaluated on test mask
# ==========================================================

df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# ----------------------------------------------------------
# Ensure post_id string
# ----------------------------------------------------------
df_all["post_id"] = df_all["post_id"].astype(str)

# ==========================================================
# 2. REBUILD POST GRAPH ON FULL DATA
# same topic + same platform + within 3h
# ==========================================================

df_all["time_dt"] = pd.to_datetime(df_all["timestamp"], errors="coerce")
df_all = df_all.dropna(subset=["time_dt"])

G = nx.Graph()
G.add_nodes_from(df_all["post_id"])

TIME_WINDOW_HOURS = 3

groups = df_all.groupby(["topic", "platform"])

for _, group in groups:

    group = group.sort_values("time_dt")

    ids = group["post_id"].values
    times = group["time_dt"].values

    n = len(group)

    for i in range(n):
        j = i + 1

        while j < n:

            diff = (times[j] - times[i]) / pd.Timedelta(hours=1)

            if diff <= TIME_WINDOW_HOURS:
                G.add_edge(ids[i], ids[j])
                j += 1
            else:
                break

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# ==========================================================
# 3. BUILD NODE FEATURES
# ==========================================================

num_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]

cat_cols = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

# Fill missing
df_all[num_cols] = df_all[num_cols].fillna(0)
df_all[cat_cols] = df_all[cat_cols].fillna("unknown")

# Scale numeric
scaler = StandardScaler()
X_num = scaler.fit_transform(df_all[num_cols])

# One-hot categorical
X_cat = pd.get_dummies(df_all[cat_cols], drop_first=False).values

# Final feature matrix
X = np.hstack([X_num, X_cat])

# ==========================================================
# 4. ALIGN GRAPH NODE ORDER
# ==========================================================

node_order = list(G.nodes())

df_all = df_all.set_index("post_id").loc[node_order].reset_index()

# rebuild X in same order
X_num = scaler.fit_transform(df_all[num_cols])
X_cat = pd.get_dummies(df_all[cat_cols], drop_first=False).values
X = np.hstack([X_num, X_cat])

y = df_all["label"].values

# ==========================================================
# 5. CONVERT TO PYG DATA
# ==========================================================

data = from_networkx(G)

data.x = torch.tensor(X, dtype=torch.float)
data.y = torch.tensor(y, dtype=torch.long)

# ==========================================================
# 6. TRAIN / TEST MASK
# based on original split
# ==========================================================

is_train = df_all["post_id"].isin(df_train["post_id"])
is_test  = df_all["post_id"].isin(df_test["post_id"])

data.train_mask = torch.tensor(is_train.values, dtype=torch.bool)
data.test_mask  = torch.tensor(is_test.values, dtype=torch.bool)

# ==========================================================
# 7. DEFINE GCN MODEL
# ==========================================================

class GCN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 32)
        self.fc = torch.nn.Linear(32, 2)

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.fc(x)

        return x

# ==========================================================
# 8. TRAIN
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCN(data.x.shape[1]).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(1, 101):

    model.train()
    optimizer.zero_grad()

    out = model(data)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")

# ==========================================================
# 9. EVALUATE
# ==========================================================

model.eval()

with torch.no_grad():

    logits = model(data)
    prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

pred = (prob >= 0.5).astype(int)

y_true = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask.cpu().numpy()]
y_prob = prob[data.test_mask.cpu().numpy()]

print("===== GCN RESULT =====")
print(classification_report(y_true, y_pred))

auc = roc_auc_score(y_true, y_prob)
print("ROC-AUC Score:", round(auc,4))

Nodes: 150000
Edges: 518562
Epoch 010 | Loss 0.5085
Epoch 020 | Loss 0.5062
Epoch 030 | Loss 0.5017
Epoch 040 | Loss 0.5012
Epoch 050 | Loss 0.5005
Epoch 060 | Loss 0.5004
Epoch 070 | Loss 0.5003
Epoch 080 | Loss 0.5001
Epoch 090 | Loss 0.4999
Epoch 100 | Loss 0.4996
===== GCN RESULT =====
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     24005
           1       0.00      0.00      0.00      5925

    accuracy                           0.80     29930
   macro avg       0.40      0.50      0.45     29930
weighted avg       0.64      0.80      0.71     29930

ROC-AUC Score: 0.5296


c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [69]:
num_neg = (data.y[data.train_mask] == 0).sum().item()
num_pos = (data.y[data.train_mask] == 1).sum().item()

weights = torch.tensor([1.0, num_neg / num_pos], dtype=torch.float).to(device)

loss = F.cross_entropy(
    out[data.train_mask],
    data.y[data.train_mask],
    weight=weights
)

In [70]:
pred = (prob >= 0.35).astype(int)

In [71]:
# ==========================================================
# 9. EVALUATE
# ==========================================================

model.eval()

with torch.no_grad():

    logits = model(data)
    prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

pred = (prob >= 0.5).astype(int)

y_true = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask.cpu().numpy()]
y_prob = prob[data.test_mask.cpu().numpy()]

print("===== GCN RESULT =====")
print(classification_report(y_true, y_pred))

auc = roc_auc_score(y_true, y_prob)
print("ROC-AUC Score:", round(auc,4))

===== GCN RESULT =====
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     24005
           1       0.00      0.00      0.00      5925

    accuracy                           0.80     29930
   macro avg       0.40      0.50      0.45     29930
weighted avg       0.64      0.80      0.71     29930

ROC-AUC Score: 0.5296


c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\khoav\anaconda3\envs\graphenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [72]:
# ==========================================================
# GCN v2 RESCUE VERSION
# Weighted loss + Better architecture + Threshold tuning
# Replace TRAIN + EVAL section only
# ==========================================================

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import numpy as np

# ==========================================================
# 1. BETTER MODEL
# ==========================================================

class BetterGCN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, 128)
        self.conv2 = GCNConv(128, 64)
        self.conv3 = GCNConv(64, 32)

        self.fc = torch.nn.Linear(32, 2)

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.35, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.25, training=self.training)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        x = self.fc(x)

        return x

# ==========================================================
# 2. DEVICE
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BetterGCN(data.x.shape[1]).to(device)
data = data.to(device)

# ==========================================================
# 3. CLASS WEIGHTS
# ==========================================================

train_y = data.y[data.train_mask]

num_neg = (train_y == 0).sum().item()
num_pos = (train_y == 1).sum().item()

weights = torch.tensor(
    [1.0, num_neg / num_pos],
    dtype=torch.float
).to(device)

print("Class weights:", weights)

# ==========================================================
# 4. OPTIMIZER
# ==========================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=1e-4
)

# ==========================================================
# 5. TRAIN
# ==========================================================

for epoch in range(1, 151):

    model.train()
    optimizer.zero_grad()

    out = model(data)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=weights
    )

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")

# ==========================================================
# 6. EVALUATE
# ==========================================================

model.eval()

with torch.no_grad():

    logits = model(data)
    prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

# test only
mask = data.test_mask.cpu().numpy()

y_true = data.y[data.test_mask].cpu().numpy()
y_prob = prob[mask]

# ==========================================================
# 7. AUTO FIND BEST THRESHOLD
# ==========================================================

best_t = 0.50
best_f1 = 0

for t in np.arange(0.20, 0.81, 0.02):

    pred = (y_prob >= t).astype(int)
    score = f1_score(y_true, pred)

    if score > best_f1:
        best_f1 = score
        best_t = t

print("Best threshold:", round(best_t,2))
print("Best F1:", round(best_f1,4))

# ==========================================================
# 8. FINAL RESULT
# ==========================================================

y_pred = (y_prob >= best_t).astype(int)

print("===== GCN v2 RESULT =====")
print(classification_report(y_true, y_pred))

auc = roc_auc_score(y_true, y_prob)
print("ROC-AUC Score:", round(auc,4))

Class weights: tensor([1.0000, 3.9898])
Epoch 010 | Loss 0.6915
Epoch 020 | Loss 0.6902
Epoch 030 | Loss 0.6893
Epoch 040 | Loss 0.6879
Epoch 050 | Loss 0.6862
Epoch 060 | Loss 0.6847
Epoch 070 | Loss 0.6827
Epoch 080 | Loss 0.6800
Epoch 090 | Loss 0.6781
Epoch 100 | Loss 0.6774
Epoch 110 | Loss 0.6744
Epoch 120 | Loss 0.6721
Epoch 130 | Loss 0.6701
Epoch 140 | Loss 0.6686
Epoch 150 | Loss 0.6662
Best threshold: 0.22
Best F1: 0.3315
===== GCN v2 RESULT =====
              precision    recall  f1-score   support

           0       0.89      0.01      0.02     24005
           1       0.20      0.99      0.33      5925

    accuracy                           0.21     29930
   macro avg       0.54      0.50      0.18     29930
weighted avg       0.75      0.21      0.08     29930

ROC-AUC Score: 0.516
